# Intelligent Login Risk Analyzer  
## Notebook 03: Feature Engineering

This notebook focuses on transforming the dataset into a structured format suitable for machine learning.

We will:
- clean and organize data
- create final feature set
- encode categorical variables
- prepare dataset for modeling

## Step 1: Import Required Libraries

We import necessary libraries for data processing and feature engineering.

In [1]:
import pandas as pd
import numpy as np

## Step 2: Load Dataset

We load the dataset and ensure all previous transformations are applied.

In [3]:
df = pd.read_excel('../data/dummy_login_dataset_300_rows.xlsx')
df.head()

,user,time,ip,location,device,status
0,user15,23:27,111.63.15.123,India,Edge,failed
1,user6,04:44,160.179.176.83,Rajshahi,Chrome,failed
2,user1,04:27,22.83.109.113,Sylhet,Firefox,failed
3,user14,17:16,15.4.242.67,Khulna,Chrome,success
4,user9,08:56,216.156.140.216,USA,Mobile,failed


## Step 3: Recreate Engineered Features

We recreate key features from Notebook 02 to ensure this notebook is self-contained.

In [4]:
# Convert time to datetime
df['time'] = pd.to_datetime(df['time'], format='%H:%M')

# Extract hour
df['hour'] = df['time'].dt.hour

# Rename column for clarity
df.rename(columns={'device': 'device_or_browser'}, inplace=True)

# Failed login feature
df['is_failed'] = df['status'].apply(lambda x: 1 if x == 'failed' else 0)

# Night login feature
df['is_night_login'] = df['hour'].apply(lambda x: 1 if 0 <= x <= 5 else 0)

# High-risk location
high_risk_locations = ['Russia']
df['is_high_risk_location'] = df['location'].apply(
    lambda x: 1 if x in high_risk_locations else 0
)

# Suspicious device
df['is_suspicious_device'] = df['device_or_browser'].apply(
    lambda x: 1 if x == 'Mobile' else 0
)

## Step 4: Encode Categorical Variables

Machine learning models require numerical input, so we encode categorical features such as user, location, and device.

In [5]:
# Encode categorical columns
df_encoded = pd.get_dummies(df, columns=['user', 'location', 'device_or_browser'], drop_first=True)

df_encoded.head()

,time,ip,status,hour,is_failed,is_night_login,is_high_risk_location,is_suspicious_device,user_user10,user_user11,...,location_India,location_Khulna,location_Rajshahi,location_Russia,location_Sylhet,location_UK,location_USA,device_or_browser_Edge,device_or_browser_Firefox,device_or_browser_Mobile
0,1900-01-01 23:27:00,111.63.15.123,failed,23,1,0,0,0,False,False,...,True,False,False,False,False,False,False,True,False,False
1,1900-01-01 04:44:00,160.179.176.83,failed,4,1,1,0,0,False,False,...,False,False,True,False,False,False,False,False,False,False
2,1900-01-01 04:27:00,22.83.109.113,failed,4,1,1,0,0,False,False,...,False,False,False,False,True,False,False,False,True,False
3,1900-01-01 17:16:00,15.4.242.67,success,17,0,0,0,0,False,False,...,False,True,False,False,False,False,False,False,False,False
4,1900-01-01 08:56:00,216.156.140.216,failed,8,1,0,0,1,False,False,...,False,False,False,False,False,False,True,False,False,True


## Step 5: Select Final Feature Set

We select relevant features for anomaly detection.

In [6]:
features = [
    'is_failed',
    'is_night_login',
    'is_high_risk_location',
    'is_suspicious_device'
]

# Add encoded columns
encoded_cols = [col for col in df_encoded.columns if col.startswith('user_') 
                or col.startswith('location_') 
                or col.startswith('device_or_browser_')]

final_features = features + encoded_cols

X = df_encoded[final_features]

X.head()

,is_failed,is_night_login,is_high_risk_location,is_suspicious_device,user_user10,user_user11,user_user12,user_user13,user_user14,user_user15,...,location_India,location_Khulna,location_Rajshahi,location_Russia,location_Sylhet,location_UK,location_USA,device_or_browser_Edge,device_or_browser_Firefox,device_or_browser_Mobile
0,1,0,0,0,False,False,False,False,False,True,...,True,False,False,False,False,False,False,True,False,False
1,1,1,0,0,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
2,1,1,0,0,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,True,False
3,0,0,0,0,False,False,False,False,True,False,...,False,True,False,False,False,False,False,False,False,False
4,1,0,0,1,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,True


## Step 6: Feature Summary

We review the final dataset structure to confirm readiness for modeling.

In [7]:
print("Feature dataset shape:", X.shape)
X.info()

Feature dataset shape: (300, 34)
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 34 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   is_failed                  300 non-null    int64
 1   is_night_login             300 non-null    int64
 2   is_high_risk_location      300 non-null    int64
 3   is_suspicious_device       300 non-null    int64
 4   user_user10                300 non-null    bool 
 5   user_user11                300 non-null    bool 
 6   user_user12                300 non-null    bool 
 7   user_user13                300 non-null    bool 
 8   user_user14                300 non-null    bool 
 9   user_user15                300 non-null    bool 
 10  user_user16                300 non-null    bool 
 11  user_user17                300 non-null    bool 
 12  user_user18                300 non-null    bool 
 13  user_user19                300 non-null    bool 
 14  user

## Step 7: Save Prepared Dataset

We save the processed dataset for use in model training.

In [8]:
X.to_csv('../outputs/processed_features.csv', index=False)

## Summary

In this notebook, we:
- recreated important features
- encoded categorical variables
- prepared a machine-learning-ready dataset
- saved the processed data for model training

This dataset will be used in the next stage: anomaly detection modeling.